# 1.3. Thu thập tài liệu tham khảo từ Semantic Scholar

## 1. Kết hợp truy vấn lô, phân trang đầy đủ và bảo toàn mã arXiv trước năm 2007

- **Truy vấn lô (`POST /graph/v1/paper/batch`)**: Gửi tối đa 50 bài mỗi lượt (nhịp nghỉ 5,0 giây, tự động chia đôi lô nếu phản hồi vượt 10 MB hoặc gặp mã 413) với danh sách trường phẳng chuẩn và đối chiếu ngược `externalIds.ArXiv` để tránh ánh xạ nhầm bài báo.
- **Phân trang theo `referenceCount` và cứu hộ từ tệp nguồn**: Khi danh sách trả về ít hơn `referenceCount`, hệ thống tự động gọi phân trang `GET /paper/arXiv:<id>/references?limit=1000` đến hết con trỏ `next`. Biểu thức chính quy nhận diện đầy đủ cả định dạng mã arXiv cũ trước năm 2007 (ví dụ `hep-th/9901001`) và chỉ giữ `publicationDate` đúng định dạng `YYYY-MM-DD`. Các bài báo mới chưa kịp lập chỉ mục trên Semantic Scholar được trích xuất bổ sung từ tệp `.bib`/`.bbl` thực tế trong bài (`bib_fallback`).

## 2. Thống kê mật độ trích dẫn và độ đầy đủ của các trường thông tin

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
df_ref_kpi = pd.DataFrame([
    ("Bài báo có kết quả S2 xác minh hoàn chỉnh (success và empty)", f"{fmt_int(m1['verified_reference_papers'])} / 25.000", fmt_pct(m1["verified_reference_papers"] / scope["papers"], 2, True)),
    ("Bài báo có dữ liệu tham khảo hợp lệ (gồm cả partial và bib_fallback)", f"{fmt_int(m1['verified_reference_papers'] + m1['reference_status'].get('partial', 0) + m1['reference_status'].get('bib_fallback', 0))} / 25.000", fmt_pct((m1["verified_reference_papers"] + 93 + 14) / scope["papers"], 2, True)),
    ("Số trích dẫn trung bình mỗi bài đã xác minh (tầng working)", f"{fmt_num(m1['mean_references_verified_papers'], 2)} trích dẫn/bài", f"Tổng {fmt_int(m1['working_reference_rows'])} dòng (0 mồ côi)"),
    ("Số trích dẫn có đích arXiv trung bình mỗi bài (bảng references)", f"{fmt_num(m1['arxiv_reference_rows'] / scope['papers'], 2)} cạnh arXiv/bài", f"{fmt_int(m1['arxiv_reference_rows'])} cạnh ({fmt_pct(m1['arxiv_reference_rows']/m1['working_reference_rows'], 2, True)})"),
], columns=["Chỉ tiêu trích dẫn M1", "Kết quả đo", "Tỷ lệ / Quy mô"])
display(style_table(df_ref_kpi, caption="Bảng 3.1. Độ phủ và số lượng tài liệu tham khảo trung bình trên 25.000 bài báo."))

rs = pd.Series(m1["reference_status"]).sort_values(ascending=False)
rs_vi = {
    "success": "Đủ danh sách tham khảo (success)",
    "empty": "Xác nhận 0 tham khảo trên S2 (empty)",
    "bib_pending": "Tạm trích từ .bib/.bbl do giới hạn 429 cuối phiên",
    "api_error": "Lỗi tạm thời API hoặc hết lượt thử lại",
    "partial": "Lệch nhẹ bộ đếm referenceCount (partial)",
    "bib_fallback": "S2 trả 404, trích xuất từ .bib/.bbl",
    "not_indexed": "S2 trả 404 và bài chỉ có PDF (not_indexed)",
}
df_rs = pd.DataFrame({
    "Trạng thái thu thập": [rs_vi.get(str(k), str(k)) for k in rs.index],
    "Mã trạng thái": [str(k) for k in rs.index],
    "Số bài báo": rs.values,
    "Tỷ lệ (%)": 100.0 * rs.values / scope["papers"],
})
display(style_table(
    df_rs,
    caption="Bảng 3.2. Phân bố trạng thái truy vấn Semantic Scholar trên 25.000 bài báo.",
    int_cols=["Số bài báo"],
    pct_cols=["Tỷ lệ (%)"],
))

miss_r = pd.Series(m1["references_missing_fraction"])
df_miss_r = pd.DataFrame({
    "Trường dữ liệu (working references)": [str(k) for k in miss_r.index],
    "Có dữ liệu (%)": (1.0 - miss_r.values) * 100.0,
    "Khuyết (%)": miss_r.values * 100.0,
})
display(style_table(
    df_miss_r,
    caption="Bảng 3.3. Tỷ lệ có dữ liệu của từng trường trên 976.351 dòng trích dẫn.",
    pct_cols=["Có dữ liệu (%)", "Khuyết (%)"],
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8), gridspec_kw={"width_ratios": [1.15, 1.0]})
y1 = np.arange(len(df_rs))
ax1.barh(y1, df_rs["Số bài báo"].to_numpy(dtype=float), color=BOOK_PALETTE["navy"], height=0.55)
ax1.set_yticks(y1)
ax1.set_yticklabels([str(v) for v in df_rs["Mã trạng thái"]])
ax1.invert_yaxis()
ax1.set_xscale("symlog", linthresh=30)
ax1.set_xlim(0, 80000)
ax1.grid(axis="x")
ax1.set_xlabel("Số bài báo (thang symlog)")
ax1.set_title("Hình 3.1a. Trạng thái S2 theo bài báo")
for i_r, v_r in enumerate(df_rs["Số bài báo"]):
    ax1.text(float(v_r) * 1.16 + 1.5, i_r, fmt_int(v_r), va="center", fontsize=8.0)

y2 = np.arange(len(df_miss_r))
ax2.barh(y2, df_miss_r["Có dữ liệu (%)"].to_numpy(dtype=float), color=BOOK_PALETTE["steel"], height=0.52)
ax2.set_yticks(y2)
ax2.set_yticklabels([str(v) for v in df_miss_r["Trường dữ liệu (working references)"]])
ax2.invert_yaxis()
ax2.set_xlim(0, 116)
ax2.grid(axis="x")
ax2.set_xlabel("Tỷ lệ có dữ liệu (%)")
ax2.set_title("Hình 3.1b. Độ đầy đủ trường trên 976.351 dòng")
for i_m, v_m in enumerate(df_miss_r["Có dữ liệu (%)"]):
    ax2.text(float(v_m) + 1.5, i_m, fmt_pct(v_m), va="center", fontsize=8.0)
plt.tight_layout()
plt.show()

```{note}
**Giải thích nguyên nhân thực tế của các trạng thái và trường khuyết:**
- **3.245 bài `empty` (12,98%) và 18 bài HTTP 404**: Phát sinh do độ trễ bóc tách tài liệu tham khảo từ PDF của Semantic Scholar đối với các bản thảo mới nộp cuối năm 2024 và đầu năm 2025.
- **93 bài `partial` (0,37%)**: Xảy ra khi chỉ số `referenceCount` nội bộ của API lớn hơn danh sách trả về thực tế từ 1 đến 2 mục dù đã duyệt hết các trang (`next = null`); hệ thống giữ nguyên các mục thu được và đánh dấu `partial` để đảm bảo tính minh bạch.
- **Trường `ref_arxiv_id` đạt 46,72% (456.149/976.351 dòng)**: Hơn một nửa tài liệu được trích dẫn là sách chuyên khảo, tiêu chuẩn kỹ thuật hoặc bài đăng tạp chí/kỷ yếu không có bản thảo trên arXiv.
```

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 1 - Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/reference_scraper.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/reference_scraper.py).
- Mã nguồn triển khai: [`src/crawler/reference_batch.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/reference_batch.py).
- Mã nguồn triển khai: [`src/crawler/reference_retry.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/reference_retry.py).
- Mã nguồn triển khai: [`src/crawler/enrich_references.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/enrich_references.py).
- Mã nguồn triển khai: [`src/crawler/observations.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/observations.py).